# A knowledge baseline for two proteins

This notebook builds a traceable knowledge baseline for two proteins with Sabueso's public
API: triosephosphate isomerase from *Trypanosoma cruzi* (TcTIM, UniProt P52270) and from
*Homo sapiens* (HsTIM, UniProt P60174). They are used here as test systems; nothing below
depends on what they are studied for.

Every value on a card is linked to the **SourceAssertions** that support it: what each
source states, when it was retrieved, and where sources disagree. The notebook asks, in
order:

1. Which entities are these, and which other entries could be mistaken for them?
2. Who says what about them?
3. What does each card know it does not know?
4. Which experimental structures are known, and which predicted models?
5. What do sources state about their oligomer and its interface?
6. Which molecules have been measured against them, and which do both share?
7. Which publications support which statements?
8. How is a claim read in a paper recorded, and compared with the databases?
9. What do the two proteins state alike, and differently?
10. How is the baseline cited, stored and read back exactly?

It queries public web services (UniProt, RCSB PDB, PDBe-KB, InterPro, ChEMBL, UniChem,
AlphaFold DB), so the numbers reflect the date it was run. The same flow runs offline, on frozen responses,
as an acceptance test (`tests/core/test_knowledge_baseline_offline.py`).

In [1]:
from datetime import date

import pyunitwizard as puw

import sabueso
from sabueso.resolver import EntityQuery

print("Sabueso", sabueso.__version__, "| run on", date.today().isoformat())

Sabueso 0.3.0+2.g866b0ae | run on 2026-09-25


## 1. Resolution

`sabueso.resolve` takes an identifier or, as here, a name and an organism (NCBI taxonomy
id). It returns the card and the **resolution**, which says how the entity was chosen.
Nothing is chosen silently: when several entries match, the rule that decided is recorded
and the others are kept as alternatives.

The profile `structural_baseline@1` names the enrichments of a structural and chemical
baseline: all experimental structures, interface residues, ligand sites, family sites and
ChEMBL bioactivities. A profile is versioned and recorded on the card, so a card says
how it was built; options passed explicitly add to it or override it. Here the AlphaFold
DB models are added too (`predicted_structures=True`).

In [2]:
cards, resolutions = {}, {}
for label, organism in (("TcTIM", 5693), ("HsTIM", 9606)):
    query = EntityQuery(name="triosephosphate isomerase", organism=organism)
    cards[label], resolutions[label] = sabueso.resolve(
        query, profile="structural_baseline@1", predicted_structures=True
    )

for label, resolution in resolutions.items():
    print(f"{label}: {resolution.status} -> {resolution.entity_ref}")
    print("   route:", resolution.decision["route"])
    print("   rules:", resolution.decision["rules"])
    print("   alternatives kept:", len(resolution.alternatives))

TcTIM: resolved -> sabueso:protein:uniprot:P52270
   route: {'entity_type': 'protein', 'tool': 'resolve_protein_card', 'basis': 'name_and_organism'}
   rules: ['name_organism_single_match']
   alternatives kept: 0
HsTIM: resolved -> sabueso:protein:uniprot:P60174
   route: {'entity_type': 'protein', 'tool': 'resolve_protein_card', 'basis': 'name_and_organism'}
   rules: ['preference:prefer_reviewed@1']
   alternatives kept: 18


In [3]:
for finding in resolutions["HsTIM"].decision["identity_audit"]:
    if "uniprot:P60174" in finding["refs"]:
        print(
            f"   {' vs '.join(finding['refs']):32} {finding['finding']:17} {finding['basis']}"
        )

   uniprot:P60174 vs uniprot:V9HWK1 possibly_same_as  {'shared_gene_loci': [['HostDB', 'ENSG00000111669'], ['NCBI Gene', '7167']], 'sequence': 'identical'}
   uniprot:P60174 vs uniprot:Q53HE2 possibly_same_as  {'shared_gene_loci': [['NCBI Gene', '7167']], 'sequence': 'near_identical', 'differences': 1, 'length': 249}
   uniprot:P60174 vs uniprot:U3KPZ0 same_gene         {'shared_gene_loci': [['HostDB', 'ENSG00000111669']], 'lengths': [249, 113]}
   uniprot:P60174 vs uniprot:U3KPS5 same_gene         {'shared_gene_loci': [['HostDB', 'ENSG00000111669']], 'lengths': [249, 38]}
   uniprot:P60174 vs uniprot:U3KQF3 same_gene         {'shared_gene_loci': [['HostDB', 'ENSG00000111669']], 'lengths': [249, 98]}


Every source consulted leaves an outcome on the card, whether it answered or not:

For HsTIM the name matched several UniProt entries; the reviewed entry was preferred by
the named rule `prefer_reviewed@1`, and the others remain listed as alternatives.

Which of those alternatives are the same protein entered again, and which are something
else? Rule `protein_identity_audit@1` compares them by gene locus and sequence, and never
merges anything: a shared gene with an agreeing sequence is flagged `possibly_same_as`,
fragments of the gene are `same_gene`, and distinct genes of one genome would be
`distinct_genes` however similar their sequences.

In [4]:
from collections import Counter

tctim, hstim = cards["TcTIM"], cards["HsTIM"]
for label, card in cards.items():
    outcomes = Counter(
        (e["source"], e.get("data", ""), e["status"])
        for e in card.quality["enrichments"]
    )
    print(label)
    for (source, data, status), n in sorted(outcomes.items()):
        print(f"   {source:9} {data:18} {status:9} x{n}")

TcTIM
   AlphaFold DB                    added     x1
   ChEMBL                       added     x1
   InterPro                     added     x1
   PDBe-KB   interface_residues added     x1
   PDBe-KB   ligand_sites       added     x1
   RCSB PDB                     added     x7
HsTIM
   AlphaFold DB                    added     x1
   ChEMBL                       added     x1
   InterPro                     added     x1
   PDBe-KB   interface_residues added     x1
   PDBe-KB   ligand_sites       added     x1
   RCSB PDB                     added     x29


## 2. Who says what

A field holds its resolved value and the ids of the SourceAssertions that support it.
Each assertion records the source, the record, the value as the source stated it and the
retrieval date. Quantities are returned as quantities, with their unit.

In [5]:
for label, card in cards.items():
    print(
        label,
        "|",
        card.get("names.canonical_name")["value"],
        "|",
        card.get("annotations.organism")["value"],
    )
    print(
        "   length:",
        card.get("sequence.length")["value"],
        "| mass:",
        puw.convert(card.quantity("sequence.molecular_weight"), to_unit="kDa"),
    )
    print(
        "   taxon:",
        card.get("annotations.taxon_id")["value"],
        "| lineage:",
        " > ".join(card.get("annotations.lineage")["value"][-3:]),
        "| gene loci:",
        len(card.get("identifiers.gene_loci")["value"]),
    )

node = hstim.get("annotations.subunit")
print("\nHsTIM subunit:", node["value"])
for sa_id in node["source_assertion_ids"]:
    assertion = hstim.source_assertion_store.get(sa_id)
    print(
        "   stated by",
        assertion["source"]["name"],
        assertion["source"]["record_id"],
        "retrieved",
        assertion["retrieved_at"],
    )
    print("   evidence (ECO):", assertion["source_metadata"]["eco"])

TcTIM | Triosephosphate isomerase, glycosomal | Trypanosoma cruzi
   length: 251 | mass: 27.329 kilodalton
   taxon: 5693 | lineage: Trypanosomatidae > Trypanosoma > Schizotrypanum | gene loci: 11
HsTIM | Triosephosphate isomerase | Homo sapiens
   length: 249 | mass: 26.669 kilodalton
   taxon: 9606 | lineage: Catarrhini > Hominidae > Homo | gene loci: 2

HsTIM subunit: ['Homodimer']
   stated by UniProt P60174 retrieved 2026-09-25T10:39:10+00:00
   evidence (ECO): [{'code': 'ECO:0000255', 'source': 'PROSITE-ProRule', 'id': 'PRU10127'}, {'code': 'ECO:0000269', 'source': 'PubMed', 'id': '18562316'}, {'code': 'ECO:0000269', 'source': 'PubMed', 'id': '8061610'}]


## 3. What each card knows it does not know

`Card.knowledge_state()` tells apart, per area and source, what is known, what sources
disagree about, what a consulted source does not state, what was not queried, and what
could not be retrieved. An absence is a fact about a source at its release, never
evidence against anything.

In [6]:
for label, card in cards.items():
    rows = card.knowledge_state()["rows"]
    print(label, dict(Counter(r["state"] for r in rows)))
    for r in rows:
        if r["state"] in ("not_stated", "not_queried", "unavailable", "conflicting"):
            print(
                f"   {r['state']:12} {r['area']:42} {r['source']:12} {r['release'] or ''}"
            )

TcTIM {'known': 27, 'not_stated': 11, 'not_queried': 1}
   not_stated   annotations.disease                        UniProt      120
   not_stated   annotations.function                       UniProt      120
   not_stated   annotations.polymorphism                   UniProt      120
   not_stated   annotations.ptm                            UniProt      120
   not_stated   annotations.tissue_specificity             UniProt      120
   not_stated   features_positional.disulfide_bond         UniProt      120
   not_stated   features_positional.glycosylation          UniProt      120
   not_stated   features_positional.modified_residue       UniProt      120
   not_stated   features_positional.mutagenesis            UniProt      120
   not_stated   features_positional.natural_variant        UniProt      120
   not_stated   relationships.interacts_with               UniProt      120
   not_queried  relationships.functionally_associated_with STRING       
HsTIM {'known': 32, 'not_stated': 6

## 4. Experimental structures and predicted models

A protein is one entity with many structures; no single PDB entry *is* the protein.
`Card.structures()` lists them with method, resolution and how much of the sequence each
one covers. Fragments and peptides are left out by default, and the exclusion is
reported.

In [7]:
for label, card in cards.items():
    view = card.structures()
    print(
        f"{label}: {len(view['items'])} structures; fragments left out: {view['excluded']}"
    )
    for item in view["items"][:5]:
        resolution = item["resolution"]
        print(
            f"   {item['structure_ref']:9} {item['method']:6} "
            f"{str(resolution) if resolution is not None else '-':>18} "
            f"{item['coverage_class']:12} {item['sources']}"
        )
    print("   ...")

TcTIM: 7 structures; fragments left out: []
   pdb:1CI1  X-ray        2.0 angstrom full_length  ['RCSB PDB', 'UniProt']
   pdb:1SUX  X-ray        2.0 angstrom full_length  ['RCSB PDB', 'UniProt']
   pdb:1TCD  X-ray       1.83 angstrom full_length  ['RCSB PDB', 'UniProt']
   pdb:2OMA  X-ray       2.15 angstrom full_length  ['RCSB PDB', 'UniProt']
   pdb:2V5B  X-ray        2.0 angstrom full_length  ['RCSB PDB', 'UniProt']
   ...
HsTIM: 24 structures; fragments left out: ['pdb:1KLG', 'pdb:1KLU', 'pdb:2IAM', 'pdb:2IAN', 'pdb:4E41']
   pdb:1HTI  X-ray        2.8 angstrom full_length  ['RCSB PDB', 'UniProt']
   pdb:1WYI  X-ray        2.2 angstrom full_length  ['RCSB PDB', 'UniProt']
   pdb:2JK2  X-ray        1.7 angstrom full_length  ['RCSB PDB', 'UniProt']
   pdb:2VOM  X-ray       1.85 angstrom full_length  ['RCSB PDB', 'UniProt']
   pdb:4BR1  X-ray        1.9 angstrom full_length  ['RCSB PDB', 'UniProt']
   ...


Predicted models are kept apart: `Card.structures()` never counts them.
`Card.predicted_structures()` lists them with their version, confidence (mean pLDDT),
coverage, and whether the model is of the entry's current sequence. AlphaFold DB also
models isoforms; such a model names its isoform and says nothing about the entry's
coverage.

In [8]:
for label, card in cards.items():
    for model in card.predicted_structures()["items"]:
        what = (
            f"isoform {model['isoform']}"
            if model["isoform"]
            else (
                f"coverage {model['coverage']}, current sequence: {model['sequence_matches']}"
            )
        )
        print(
            f"{label}: {model['model_ref']:26} v{model['model_version']} "
            f"mean pLDDT {model['mean_plddt']:6} | {what}"
        )

TcTIM: alphafold:AF-P52270-F1     v6 mean pLDDT  97.31 | coverage 1.0, current sequence: True
HsTIM: alphafold:AF-P60174-3-F1   v6 mean pLDDT  89.88 | isoform P60174-3
HsTIM: alphafold:AF-P60174-4-F1   v6 mean pLDDT  97.31 | isoform P60174-4
HsTIM: alphafold:AF-P60174-F1     v6 mean pLDDT  96.69 | coverage 1.0, current sequence: True


## 5. Oligomer and interface

`Card.oligomer()` puts side by side what each source states about quaternary structure:
UniProt's subunit statement, the assemblies RCSB annotates for each structure, the
interface residues PDBe-KB derives from the structures (per partner chain), and the
dimer-interface site of the family model (CDD, through InterPro).

Each PDBe-KB "partner" is classified, structure by structure, so that a chimera or a
peptide complex is not mistaken for a complex of the protein.

In [9]:
for label, card in cards.items():
    view = card.oligomer()
    print(label, "| UniProt subunit:", [s["text"] for s in view["subunit"]])
    states = Counter(
        a["oligomeric_state"] for e in view["assemblies"] for a in e["assemblies"]
    )
    print("   RCSB assemblies:", dict(states))
    for interface in view["interfaces"]:
        print(
            f"   {interface['partner_ref']:38} {interface['class']:17} "
            f"{len(interface['positions']):3} residues"
        )
    for row in view["agreement"]:
        print(
            f"   {row['signature']} {row['family_site']}: {len(row['both'])} positions also"
            f" observed, family only {row['family_only']}, observed only "
            f"{len(row['observed_only'])}"
        )

TcTIM | UniProt subunit: ['Homodimer']
   RCSB assemblies: {'Homo 2-mer': 8, 'Monomer': 1}
   uniprot:P52270                         homomeric          36 residues
   uniprot:P04789                         chimera            32 residues
   cd00311 dimer interface: 12 positions also observed, family only [53], observed only 24
HsTIM | UniProt subunit: ['Homodimer']
   RCSB assemblies: {'Homo 2-mer': 29, 'Hetero 4-mer': 2, 'Homo 4-mer': 1, 'Hetero 5-mer': 7}
   uniprot:P60174                         homomeric          36 residues
   uniprot:P01903                         fragment_complex   14 residues
   uniprot:P01911                         fragment_complex   12 residues
   pdbe_kb.partner:TR-alpha light chain   fragment_complex    6 residues
   uniprot:P01848                         fragment_complex    6 residues
   pdbe_kb.partner:TR-beta light chain    fragment_complex    5 residues
   uniprot:P01850                         fragment_complex    4 residues
   cd00311 dimer interface: 

In TcTIM, PDBe-KB lists TbTIM (P04789) as a partner only because PDB entry 3Q37 is a
TcTIM/TbTIM chimera: one polymer entity mapped to both proteins. In HsTIM, the HLA-DR and
T-cell receptor chains come from complexes with a TIM peptide. Neither is a complex of
the enzyme, and the classes say so.

## 6. Ligands and bioactivities

ChEMBL measurements are `has_bioactivity` relationships, one per measurement.
`Card.bioactivities()` derives an activity class for each from explicit thresholds (a
rule, recorded with the view), and leaves out measurements ChEMBL assigned to the target
by homology unless asked.

In [10]:
for label, card in cards.items():
    view = card.bioactivities()
    print(
        label,
        "|",
        len(view["items"]),
        "molecules;",
        "classes:",
        dict(Counter(i["class"] for i in view["items"])),
        "| left out (not direct):",
        len(view["excluded"]),
    )
print(
    "rule:",
    view["classification"]["rule"],
    "| checks:",
    [c["rule"] for c in view["checks"]],
)

TcTIM | 256 molecules; classes: {'active': 8, 'weak': 26, 'inactive': 201, 'inconclusive': 21} | left out (not direct): 0


HsTIM | 26 molecules; classes: {'active': 11, 'weak': 2, 'inactive': 12, 'not_determined': 1} | left out (not direct): 8
rule: bioactivity_class@3 | checks: ['pchembl_consistency@1', 'unit_scale_discrepancy@1']


A **ligand deck** turns the measured molecules and the ligands the structures were
determined to study into small-molecule cards, one per standard InChIKey. Comparing two
proteins' decks shows which molecules both share.

In [11]:
decks = {label: sabueso.ligand_deck(card) for label, card in cards.items()}
for label, deck in decks.items():
    print(label, "|", len(deck.cards), "molecules |", deck.meta["sources"])

comparison = tctim.compare_ligands(decks["TcTIM"], hstim, decks["HsTIM"])
print("\nshared by both:", len(comparison["shared"]))

# "label" is the molecule's name, else its ChEMBL id, else its PDB component code.
for item in comparison["shared"][:6]:
    mine, theirs = item["self"]["bioactivity"], item["other"]["bioactivity"]
    print(f"   {item['label']:30} TcTIM: {mine['class']:10} HsTIM: {theirs['class']}")

TcTIM | 256 molecules | [{'source': 'ChEMBL', 'status': 'added', 'requested': 256, 'found': 256, 'missing': []}, {'source': 'PDB CCD', 'status': 'added', 'requested': 1, 'found': 1, 'missing': []}]
HsTIM | 36 molecules | [{'source': 'ChEMBL', 'status': 'added', 'requested': 33, 'found': 33, 'missing': []}, {'source': 'PDB CCD', 'status': 'added', 'requested': 3, 'found': 3, 'missing': []}]

shared by both: 14
   CHEMBL4576576                  TcTIM: weak       HsTIM: inactive
   CHEMBL567497                   TcTIM: inactive   HsTIM: inactive
   CHEMBL4462472                  TcTIM: weak       HsTIM: inactive
   CHEMBL1630897                  TcTIM: weak       HsTIM: inactive
   CHEMBL4443832                  TcTIM: inactive   HsTIM: inactive
   METHYLBREVIFOLIN CARBOXYLATE   TcTIM: active     HsTIM: inactive


## 7. Literature

`Card.literature()` gathers, per publication, the sources that cite it and what for (the
scope of a UniProt reference), the structures whose primary citation it is, and the
statements whose evidence names it. It does not read papers.

In [12]:
for pub in tctim.literature()["publications"]:
    scope = [s for cited in pub["cited_by"] for s in cited["scope"]]
    print(pub["publication_ref"], pub["year"], "|", (pub["title"] or "")[:70])
    if scope:
        print("     cited by UniProt for:", scope)
    if pub["primary_citation_of"]:
        print("     primary citation of:", pub["primary_citation_of"])

pubmed:9108237 1997 | Cloning, expression, purification and characterization of triosephosph
     cited by UniProt for: ['NUCLEOTIDE SEQUENCE [GENOMIC DNA]']
pubmed:9761683 1998 | Differences in the intersubunit contacts in triosephosphate isomerase 
     cited by UniProt for: ['X-RAY CRYSTALLOGRAPHY (1.83 ANGSTROMS)', 'HOMODIMERIZATION']
     primary citation of: ['pdb:1TCD']
pubmed:10468562 1999 | Crystal structure of triosephosphate isomerase from Trypanosoma cruzi 
     cited by UniProt for: ['X-RAY CRYSTALLOGRAPHY (2.0 ANGSTROMS)', 'HOMODIMERIZATION']
     primary citation of: ['pdb:1CI1']
pubmed:15321726 2004 | Inactivation of triosephosphate isomerase from Trypanosoma cruzi by an
     primary citation of: ['pdb:1SUX']
pubmed:17989778 2007 | Perturbation of the Dimer Interface of Triosephosphate Isomerase and i
     primary citation of: ['pdb:2OMA']
pubmed:19121704 2008 | The Monomerization of Triosephosphate Isomerase from Trypanosoma Cruzi
     primary citation of: ['pdb:2V5B']

## 8. A claim read in a paper

Much knowledge exists only in papers. When a person (or, later, an agent) reads one, the
claim is recorded as a **curated literature assertion**: a SourceAssertion whose source is
the publication, with the curator, where in the paper it is stated, and optionally a short
quote. Sabueso checks its shape against the field, and compares it with what databases
state. It never gives it priority, and never discards anything.

The title of PubMed 18562316 relates the HsTIM deficiency mutation E104D (E105D in UniProt
numbering, which counts the initial methionine) to a conserved water network at the dimer
interface. UniProt describes the same variant differently:

In [13]:
from sabueso.core.card import Card

hstim_as_built = Card.from_dict(
    hstim.to_dict()
)  # kept, to cite this state in section 10

variant = next(
    i
    for i in hstim.get("features_positional.natural_variant")["value"]
    if i["location"]["sequence"]["start"] == 105
)
print("UniProt:", variant["description"])

record = hstim.add_literature_assertion(
    "features_positional.natural_variant",
    {
        "start": 105,
        "substitution": {"original": "E", "alternatives": ["D"]},
        "description": "alters a conserved water network at the dimer interface",
    },
    publication="pubmed:18562316",
    curator="showcase",
    locator="Title",
)
print("\noutcome:", record["outcome"])

UniProt: in TPID; no effect on triose-phosphate isomerase activity; changed protein homodimerization activity; the homodimer stability is temperature-dependent and affects the triose-phosphate isomerase activity; dbSNP:rs121964845

outcome: differs


The outcome is `differs`: the same item (position 105, E to D) is stated differently.
Sabueso cannot tell whether two texts contradict each other, so it flags the difference
for a reader instead of judging it: it is recorded in `quality.conflicts`, a warning is
shown, and both statements stay on the card. A free-text field such as the subunit
statement is recorded but not compared:

In [14]:
record = hstim.add_literature_assertion(
    "annotations.subunit",
    "Homodimer",
    publication="pubmed:8061610",
    curator="showcase",
    locator="Abstract",
)
print("outcome:", record["outcome"])

print("\nconflicts:", [(c["field"], c["type"]) for c in hstim.quality["conflicts"]])
for pub in hstim.literature()["publications"]:
    for curated in pub["curated"]:
        print(
            pub["publication_ref"], "|", curated["field_path"], "|", curated["outcome"]
        )

outcome: not_compared

conflicts: [('features_positional.natural_variant', 'curated_difference')]
pubmed:8061610 | annotations.subunit | not_compared
pubmed:18562316 | features_positional.natural_variant | differs


How a claim bears on a project's hypotheses is not Sabueso's to record: that is
**Evidence**, and it belongs to Nextia. A SourceAssertion says what a source states.

## 9. The two proteins side by side

`Card.compare_knowledge` says what two cards both state, what only one states, and what
they state differently. Positions are compared only through a residue mapping, because
the same number in two entries is not the same residue. In a workflow the mapping comes
from an alignment (MolSysMT); here it maps the four annotated catalytic and substrate
positions, which UniProt numbers 96 and 168 in TcTIM and 96 and 166 in HsTIM.

In [15]:
diff = tctim.compare_knowledge(hstim, residue_map={12: 12, 14: 14, 96: 96, 168: 166})
for path in (
    "features_positional.active_site",
    "features_positional.binding_site",
    "annotations.function",
    "annotations.subunit",
):
    print(
        f"{path:34}",
        {k: v for k, v in diff["fields"][path].items() if k in ("status", "reason")},
        "| both:",
        len(diff["fields"][path].get("both", [])),
    )
for predicate in ("classified_in", "annotated_with", "has_bioactivity"):
    r = diff["relationships"][predicate]
    print(
        f"{predicate:18} both {len(r['both']):3}  only TcTIM {len(r['only_self']):3}"
        f"  only HsTIM {len(r['only_other']):3}"
    )

features_positional.active_site    {'status': 'same'} | both: 2
features_positional.binding_site   {'status': 'same'} | both: 2
annotations.function               {'status': 'only_other'} | both: 0
annotations.subunit                {'status': 'not_compared', 'reason': 'free text'} | both: 0
classified_in      both  13  only TcTIM   0  only HsTIM   2
annotated_with     both   5  only TcTIM   2  only HsTIM   8
has_bioactivity    both  14  only TcTIM 242  only HsTIM  19


## 10. Cite, store and read back exactly

A card's **snapshot id** is the content address of its exact state. A knowledge store
keeps every state it is given, and a pinned reference resolves to that state or fails;
it never returns another. Saving a changed card adds a revision, and earlier references
keep resolving. These reference forms are provisional until they are agreed across MOLI
(uibcdf/moli#3).

In [16]:
import tempfile
from pathlib import Path

from sabueso.core.deck import Deck

with tempfile.TemporaryDirectory() as tmp:
    store = sabueso.KnowledgeStore(Path(tmp) / "baseline.db")
    before = store.save(hstim_as_built, note="as built")
    after = store.save(hstim, note="with the curated claims of section 8")
    print("before:", before[:70] + "...")
    print("after: ", after[:70] + "...")
    print("revisions:", [(h["revision"], h["note"]) for h in store.history(hstim.id)])
    old = store.load(before)
    print(
        "curated claims in the pinned earlier state:",
        len(old.quality.get("curation", [])),
    )

    store.save(tctim)
    shared = store.relationships(
        object_ref="interpro:IPR000652", predicate="classified_in"
    )
    print(
        "cards classified in the TIM domain:",
        sorted({r["card"].split("@")[0] for r in shared}),
    )

    deck_ref = store.save_deck(
        Deck([tctim, hstim], meta={"purpose": "showcase"}), "tims"
    )
    print("deck:", deck_ref[:60] + "...")
    print("cards of the pinned deck:", store.load_deck(deck_ref).ids())

before: sabueso:protein:uniprot:P60174@sha256:48dd8d2f4b390b21ed2618afb10dc8ae...
after:  sabueso:protein:uniprot:P60174@sha256:b8036d1c8fa627aad6fbe159ca736133...
revisions: [(1, 'as built'), (2, 'with the curated claims of section 8')]
curated claims in the pinned earlier state: 0


cards classified in the TIM domain: ['sabueso:protein:uniprot:P52270', 'sabueso:protein:uniprot:P60174']
deck: sabueso:deck:tims@sha256:ff7a9a81771d9025ffe38b98cac8e2486bb...


cards of the pinned deck: ['sabueso:protein:uniprot:P52270', 'sabueso:protein:uniprot:P60174']


## What is not here

- Nothing is computed from coordinates: interfaces and contacts are what sources state.
  Geometry is modelling, and belongs to other components (uibcdf/sabueso#30).
- Free-text claims (uibcdf/sabueso#43) and biological context such as life-cycle stage
  or essentiality (uibcdf/sabueso#60) are not structured yet.
- Sabueso also records curated ranges and uncertainties of measurements, and the residues
  a paper says a compound acts on (`add_literature_bioactivity`,
  `add_literature_engagement`); this notebook does not invent a paper to show them.
- Sabueso records what sources state; how it bears on a study is Nextia Evidence.